# Retrieval Augmented Generation (RAG) with Langchain

- The most common approach in RAG is to create dense vector representations of the knowledge base in order to retrieve text chunks that are semantically similar to a given user query. **->** (*That ability to find text based on meaning rather than exact keywords is what the sentence is describing*)


In [29]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    transformers \
    langchain_classic \
    langchain_community \
    langchain_text_splitters \
    langchain_huggingface sentence_transformers \
    langchain_chroma chromadb \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    wget
! echo "::endgroup::"

::group::Install Dependencies
Using Python 3.12.13 environment at: /usr
Resolved 137 packages in 1.13s
Checked 137 packages in 1ms
::endgroup::


In [30]:
## Get Embedding
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(
    model_name=embeddings_model_path,
)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [31]:
## DAtabase for strong embedding
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)

In [32]:
## Integrate LLM
! pip install -U "langchain[google-genai]"

In [33]:
from google.colab import userdata
from langchain.chat_models import init_chat_model
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash-lite")


In [34]:
## Import Document

In [35]:
import os
import wget

filename = 'text.txt'
url = 'https://raw.githubusercontent.com/puffer20/RAG_sample_text/refs/heads/main/text.txt'

if not os.path.isfile(filename):
  wget.download(url, out=filename)

In [36]:
## Chunk the text
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embeddings_tokenizer,
    chunk_size=100,
    chunk_overlap=0,
)
texts = text_splitter.split_documents(documents)
doc_id = 0
for text in texts:
    text.metadata["doc_id"] = (doc_id:=doc_id+1)
print(f"{len(texts)} text document chunks created")

7 text document chunks created


In [37]:
## Populate the vector database
ids = vector_db.add_documents(texts)
print(f"{len(ids)} documents added to the vector database")

7 documents added to the vector database


In [38]:
## quering
query = "When did germany surrendered unconditionally?"
docs = vector_db.similarity_search(query)
print(f"{len(docs)} documents returned")
for doc in docs:
    print(doc)
    print("=" * 80)  # Separator for clarity

4 documents returned
page_content='The Battle of the Bulge caught the Allies off guard. The German armor pushed a sixty-mile deep indentation into the American lines. At the vital road junction of Bastogne, the US 101st Airborne Division was completely surrounded, outgunned, and short on winter gear. When the German commander demanded their surrender, the American commander, Brigadier General Anthony McAuliffe, delivered a legendary one-word reply: "Nuts!"The Americans held Bastogne. By late December, the skies cleared, allowing Allied fighter-bombers to strike German fuel columns. Patton executed a brilliant tactical shift, turning his entire army ninety degrees north in the snow to relieve the city. The German offensive exhausted its last strategic oil reserves, leaving the Wehrmacht defenseless against the final Allied advance into Germany.Act V: The Agony and the End (1945)The final year of the war was an exercise in total destruction. Germany was subjected to a relentless round-th

### Automate pipeline
 User Query
    ↓
Retriever
    ↓
Relevant Documents
    ↓
Prompt Construction
    ↓
LLM
    ↓
Answer


In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Step 1: Define the prompt template
prompt_template = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}
""")

# Step 2: Define a runnable to format documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Step 3: Build the RAG chain explicitly using LCEL
# This chain will:
# 1. Retrieve documents based on the input query.
# 2. Format these documents into a single string.
# 3. Combine the formatted context string and the input query into the prompt.
# 4. Pass the final prompt message to the LLM.

rag_chain = (
    {
        "context": itemgetter("input") | vector_db.as_retriever() | format_docs, # Retrieve, then format
        "input": itemgetter("input"), # Keep the original input
    }
    | prompt_template # Apply the prompt template to get a ChatPromptValue
    | model # Invoke the LLM with the ChatPromptValue
)

In [40]:
from ibm_granite_community.notebook_utils import wrap_text

output = rag_chain.invoke({"input": query})

print(wrap_text(output.content)) # LLM output is a BaseMessage, access content

Germany surrendered unconditionally on May 8, 1945.
